# 04: upstream LEGATO, baseline vs contrastive, edge recovery and the paper's downstream metric

In [1]:
import sys, subprocess, math
from pathlib import Path
import numpy as np, torch, torch.nn.functional as F
from torch import nn, optim
from sklearn.metrics import roc_auc_score

UP = Path("../external/LEGATO").resolve()
assert subprocess.check_output(["git", "-C", str(UP), "rev-parse", "HEAD"], text=True).startswith("514224b")
assert not subprocess.check_output(["git", "-C", str(UP), "status", "--porcelain", "--untracked-files=no"], text=True).strip()
sys.path[:0] = [str(UP), str(UP / "exps" / "simulation_exp")]
from method.LEGATO import LEGATO
from method.BaseNetworks import Encoder, Decoder
import method.GraphLearner as GLmod
from simulate import gen_synthetic, get_dataloaders

<local path> FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


<local path> TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<local path> FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [2]:
orig_forward = GLmod.LearnerHead.forward

def patched_forward(self, b_het_z):
    h = self.act(torch.matmul(b_het_z, self.W))
    self.last_h = h                                   # (B, K, 100), the only addition
    b_adj = self.cos_criterion(h[:, None, :, :], h[:, :, None, :]).abs()
    return self.threshold(b_adj)

import method.GraphDataUtils as GDU
import torch_geometric.utils as PYGU

def d2s(adj, mask=None):                       # batched (B, N, N) dense -> sparse; same output as PyG, no in-place ops
    b, i, j = adj.nonzero(as_tuple=True)
    n = adj.shape[-1]
    return torch.stack([i + b * n, j + b * n]), adj[b, i, j]

a = torch.rand(4, 5, 5) * (torch.rand(4, 5, 5) > 0.4)
assert all(torch.equal(x, y) for x, y in zip(PYGU.dense_to_sparse(a), d2s(a))), "d2s differs from PyG"
GDU.dense_to_sparse = d2s; GLmod.dense_to_sparse = d2s

def build_model(K=6, d=100, hidden=60, thr=0.1):
    enc = nn.ModuleList(Encoder(d, hidden, hidden, 1) for _ in range(K))
    dec = nn.ModuleList(Decoder(hidden, d, hidden, 1) for _ in range(K))
    return LEGATO(n_views=K, n_in_feats=hidden, encoder_list=enc, decoder_list=dec,
                  pool_ratio=0.5, sparse_threshold=thr, device="cpu")

GLmod.LearnerHead.forward = patched_forward   # apply the patch

In [3]:
def true_edges_local(K):
    t = np.zeros((K, K))
    for i in range(0, K, 2): t[i, i + 1] = t[i + 1, i] = 1
    return t


In [4]:
import copy
from training_utils.losses import RegLoss

def info_nce(a, b, tau=0.1):
    a, b = F.normalize(a, dim=1), F.normalize(b, dim=1)
    return F.cross_entropy(a @ b.T / tau, torch.arange(a.shape[0]))

def contrast(model):   # mean InfoNCE over the K(K-1)/2 view pairs, on the projected vectors from the last forward pass
    h = model.graph_learner.heads[0].last_h
    K = h.shape[1]
    return torch.stack([info_nce(h[:, i], h[:, j]) for i in range(K) for j in range(i + 1, K)]).mean()

def pretrain_c(model, criterion, opt, loaders, lam, EPOCHS=1000, PATIENCE=20, seed=0, val_bn_train=False):
    torch.manual_seed(seed); np.random.seed(seed)
    best, best_state, wait = np.inf, None, 0
    for epoch in range(EPOCHS):
        model.train(); tr = n = 0
        for data in loaders["train_unlab"]:
            views = [v for v in data]
            opt.zero_grad()
            out, lat, A, misc = model(views)
            loss = criterion(views, out, lat[0], A, misc[2])[0] + lam * contrast(model)
            loss.backward(); opt.step()
            tr += loss.item() * views[0].shape[0]; n += views[0].shape[0]
        model.train() if val_bn_train else model.eval(); va = m = 0
        with torch.no_grad():
            for data in loaders["val_unlab"]:
                views = [v for v in data]
                out, lat, A, misc = model(views)
                loss = criterion(views, out, lat[0], A, misc[2])[0] + lam * contrast(model)
                va += loss.item() * views[0].shape[0]; m += views[0].shape[0]
        va /= m
        if va < best: best, best_state, wait = va, copy.deepcopy(model.state_dict()), 0
        else: wait += 1
        if wait == PATIENCE: break
    model.load_state_dict(best_state)
    return dict(epochs=epoch + 1, best_val=best)

In [5]:
from method.LinearProbe import LinearProbe
from training_utils.train_evaluate import finetune, evaluate
import tempfile, os, io, contextlib, time
import pandas as pd

def edge_auroc(model, X_lab, K, train_mode=False):
    model.train() if train_mode else model.eval()
    with torch.no_grad():
        _, _, A, _ = model([torch.tensor(x, dtype=torch.float32) for x in X_lab])
    iu = np.triu_indices(K, 1)
    return roc_auc_score(true_edges_local(K)[iu], A.mean(0).mean(0).numpy()[iu])

class ProbeBN(LinearProbe):   # the authors' probe, but the encoder keeps batch-statistics BatchNorm
    def forward(self, x):
        self.encoder.train()
        _, g, _, _ = self.encoder(x)
        return self.linear_probe(g[0].mean(dim=1))

In [6]:
def run_up(seed, lam, w=0.75, protocol="upstream", K=6, hidden=60):
    synth = gen_synthetic("local", num_views=K, corr=w, output_dim=100, num_samples=1000, seed=seed)
    X_lab = [x[800:] for x in synth[0]]
    loaders = get_dataloaders(synth, K, 64, seed=seed)
    torch.manual_seed(seed); np.random.seed(seed)
    model = build_model(K, hidden=hidden)
    opt = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-3)
    bn = protocol == "bn_train"
    r = pretrain_c(model, RegLoss(alpha=0.1, beta=0.1), opt, loaders, lam=lam, EPOCHS=300 if bn else 1000,
                   PATIENCE=10**9 if bn else 20, seed=seed, val_bn_train=bn)
    out = dict(seed=seed, lam=lam, w=w, protocol=protocol, epochs=r["epochs"],
               auc_eval=edge_auroc(model, X_lab, K), auc_bn=edge_auroc(model, X_lab, K, train_mode=True))
    f = tempfile.mktemp(suffix=".pt")
    with contextlib.redirect_stdout(io.StringIO()):
        probe = (ProbeBN if bn else LinearProbe)(encoder=model, n_in_feats=model.latent_dims, n_targets=K, freeze_encoder=False)
        finetune(probe, nn.MSELoss(), optim.Adam(probe.parameters(), lr=1e-3, weight_decay=1e-3), loaders, 1000, 20, f, seed, "cpu", True)
        probe.load_state_dict(torch.load(f)); os.remove(f)
        out["mse"] = evaluate(probe, nn.MSELoss(), loaders, clas=False, DEVICE="cpu")
    out["mse_meanpred"] = float(synth[1][800:].var(0).mean())
    return out

In [7]:
for protocol in ("upstream", "bn_train"):
    for lam in (0, 1):
        t0 = time.time(); r = run_up(0, lam, 0.75, protocol)
        print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in r.items()}, f"{time.time()-t0:.0f}s")

{'seed': 0, 'lam': 0, 'w': 0.75, 'protocol': 'upstream', 'epochs': 64, 'auc_eval': np.float64(0.417), 'auc_bn': np.float64(0.639), 'mse': 0.021, 'mse_meanpred': 0.986} 10s


{'seed': 0, 'lam': 1, 'w': 0.75, 'protocol': 'upstream', 'epochs': 27, 'auc_eval': np.float64(0.556), 'auc_bn': np.float64(0.917), 'mse': 0.195, 'mse_meanpred': 0.986} 5s


{'seed': 0, 'lam': 0, 'w': 0.75, 'protocol': 'bn_train', 'epochs': 300, 'auc_eval': np.float64(0.75), 'auc_bn': np.float64(0.458), 'mse': 0.257, 'mse_meanpred': 0.986} 33s


{'seed': 0, 'lam': 1, 'w': 0.75, 'protocol': 'bn_train', 'epochs': 300, 'auc_eval': np.float64(0.889), 'auc_bn': np.float64(1.0), 'mse': 0.253, 'mse_meanpred': 0.986} 33s


In [8]:
def sweep_up(seeds=range(5), ws=(0.25, 0.5, 0.75), lams=(0, 1), protocols=("upstream", "bn_train")):
    return pd.DataFrame([run_up(s, l, w, p) for p in protocols for w in ws for l in lams for s in seeds])

In [9]:
t0 = time.time()
df = sweep_up()
df.to_csv("../derived/nb04_sweep.csv", index=False)
print(f"{len(df)} runs in {time.time()-t0:.0f}s")

60 runs in 1213s


In [10]:
cols = ["epochs", "auc_eval", "auc_bn", "mse"]
display(df.groupby(["protocol", "w", "lam"])[cols].agg(["mean", "std"]).round(3))

for m in ["auc_eval", "auc_bn", "mse"]:
    piv = df.pivot_table(index=["protocol", "w", "seed"], columns="lam", values=m)
    d = (piv[1] - piv[0]).groupby(["protocol", "w"])
    print(m, "(lam1 - lam0), mean paired difference and number of seeds where lam1 is higher, of 5")
    display(pd.DataFrame({"mean_diff": d.mean().round(3), "seeds_lam1_higher": d.apply(lambda x: int((x > 0).sum()))}))
print("mean of always predicting the mean latent:", round(df.mse_meanpred.mean(), 3))

epochs         auc_eval        auc_bn           mse       
                    mean     std     mean    std   mean    std   mean    std
protocol w    lam                                                           
bn_train 0.25 0    300.0   0.000    0.683  0.112  0.492  0.019  0.342  0.184
              1    300.0   0.000    0.622  0.144  0.828  0.176  0.412  0.358
         0.50 0    300.0   0.000    0.650  0.130  0.500  0.000  0.375  0.131
              1    300.0   0.000    0.631  0.075  1.000  0.000  0.411  0.273
         0.75 0    300.0   0.000    0.581  0.197  0.650  0.222  0.302  0.136
              1    300.0   0.000    0.764  0.152  1.000  0.000  0.295  0.115
upstream 0.25 0     39.0   8.972    0.517  0.127  0.492  0.159  0.069  0.086
              1     36.0  12.510    0.528  0.062  0.817  0.207  0.038  0.019
         0.50 0     41.4   8.355    0.478  0.163  0.617  0.194  0.028  0.015
              1     44.2  18.308    0.628  0.046  0.978  0.050  0.019  0.007
         0.75 0     47.4  18.796    0.397  0.027  0.661  0.168  0.029  0.018
              1     46.4  16.517    0.644  0.114  0.983  0.037  0.054  0.079

auc_eval (lam1 - lam0), mean paired difference and number of seeds where lam1 is higher, of 5


mean_diff  seeds_lam1_higher
protocol w                                 
bn_train 0.25     -0.061                  1
         0.50     -0.019                  3
         0.75      0.183                  4
upstream 0.25      0.011                  2
         0.50      0.150                  4
         0.75      0.247                  5

auc_bn (lam1 - lam0), mean paired difference and number of seeds where lam1 is higher, of 5


mean_diff  seeds_lam1_higher
protocol w                                 
bn_train 0.25      0.336                  5
         0.50      0.500                  5
         0.75      0.350                  4
upstream 0.25      0.325                  4
         0.50      0.361                  5
         0.75      0.322                  5

mse (lam1 - lam0), mean paired difference and number of seeds where lam1 is higher, of 5


mean_diff  seeds_lam1_higher
protocol w                                 
bn_train 0.25      0.070                  2
         0.50      0.036                  2
         0.75     -0.007                  1
upstream 0.25     -0.031                  1
         0.50     -0.009                  1
         0.75      0.025                  3

mean of always predicting the mean latent: 0.978
